# Explicit input validation and DFT extraction

FCC ISF/ESF map to their same-type perfect subdirectories; HCP I1/I2 map to the corresponding alloy reference under PERFECT. Choose final paths explicitly, including base/new versions. Never pair independently numbered collections.

In [ ]:
from pathlib import Path
import os
import sys
HERE = Path.cwd().resolve()
MODULE_DIR = next(p for p in [HERE, HERE/'notebooks', HERE.parent/'notebooks']
                  if (p/'benchmark_workflows.py').is_file())
sys.path.insert(0,str(MODULE_DIR))
import benchmark_workflows as workflows
REPOSITORY = workflows.repo_root()

def validate_sfe_manifest(path):
    import numpy as np
    rows = workflows.table(path)
    if len({r['ID'] for r in rows}) != len(rows):
        raise ValueError('Duplicate pair IDs.')
    report=[]
    for r in rows:
        a=workflows.read_atoms({'structure_path':r['fault_path']},Path(path).parent)
        b=workflows.read_atoms({'structure_path':r['perfect_path']},Path(path).parent)
        if sorted(a.numbers)!=sorted(b.numbers) or not np.allclose(a.cell.array,b.cell.array,atol=1e-6,rtol=0):
            raise ValueError(f"Invalid composition or cell: {r['ID']}")
        if r['fault_type'] not in ['I1','I2','ISF','ESF']:
            raise ValueError(f"Unknown fault type: {r['ID']}")
        report.append({'ID':r['ID'],'fault_type':r['fault_type'],
                       'area_A2':float(np.linalg.norm(np.cross(b.cell[0],b.cell[1]))),
                       'fault_constraint_count':len(a.constraints)})
    return report
VALIDATE_INPUTS=False
if VALIDATE_INPUTS:
    workflows.save_table(REPOSITORY/'results/input_validation.csv',
                          validate_sfe_manifest(REPOSITORY/'inputs/sfe.csv'))


## Final zero-smearing DFT energy

Convergence messages are evidence to inspect, not proof of physically reliable references. No automatic path or energy selection is performed.

In [ ]:
import re
def extract_dft_energy(outcar):
    text=Path(outcar).read_text(errors='replace')
    values=re.findall(r'energy\(sigma->0\)\s*=\s*([-+0-9.eE]+)',text)
    if not values:
        raise ValueError(f'No final energy(sigma->0): {outcar}')
    return {'energy_sigma0_eV':float(values[-1]),
            'ionic_convergence_message':'reached required accuracy' in text,
            'normal_exit_message':'General timing' in text}


## Read consolidated data
PKL files contain plain Python records, not live calculator objects. HCP I1 is pending.


In [ ]:
import pickle
READ_CONSOLIDATED = False
if READ_CONSOLIDATED:
    with (REPOSITORY/'data/surface/dft/reference.pkl').open('rb') as stream:
        surface_reference = pickle.load(stream)
    print(surface_reference['dataset_status'], len(surface_reference['records']))
